# MVSR – Vorbereitung Einheit 7
## Visual Servoing: Regelkreis, Optical Flow, Interaction Matrix, IBVS und PBVS

**C++17 / MyBinder**  
**Bearbeitungszeit:** ca. 45–60 Minuten

Dieses Notebook führt von der visuellen Rückführung über Bildbewegung und Interaction Matrix zum geschlossenen Visual-Servoing-Regelkreis.

**Roter Faden:**

**visuelle Rückführung → Bildbewegung → Interaction Matrix → Regelgesetz → IBVS → PBVS**

### Lernziele

Nach Bearbeitung dieses Notebooks können Sie ...

- Visual Servoing als geschlossenen Regelkreis erklären,
- Bildbewegung als Folge einer Kamerabewegung interpretieren,
- den Zusammenhang $\dot{\mathbf{s}}=\mathbf{L}_s\mathbf{v}_c$ erklären,
- die Interaction Matrix eines Bildpunktes interpretieren,
- die lokale VS-Strecke regelungstechnisch einordnen,
- das klassische IBVS-Regelgesetz herleiten,
- einen einfachen IBVS-Regelkreis simulieren,
- den Einfluss von Verstärkung und Tiefenschätzung untersuchen,
- IBVS und PBVS anhand ihrer Trajektorien vergleichen.

### Hinweise zur Verwendung dieses Notebooks

Dieses Notebook ist für **MyBinder mit xeus-cling / C++17** vorbereitet.

Arbeiten Sie es möglichst **von oben nach unten** durch. Verändern Sie insbesondere die mit `TODO` markierten Parameter und beobachten Sie die Auswirkungen.

Die Visualisierungen werden direkt mit OpenCV erzeugt. Dadurch sind keine zusätzlichen C++-Plotbibliotheken notwendig.

> Falls der Kernel nach Änderungen an Funktionsdefinitionen Fehler wegen Mehrfachdefinitionen meldet, starten Sie den Kernel neu und führen Sie das Notebook erneut von oben aus.

## 0. Setup

In [ ]:
#include <iostream>
#include <iomanip>
#include <vector>
#include <string>
#include <cmath>
#include <algorithm>
#include <numeric>
#include <cstdlib>
#include <utility>

#if __has_include("CppInterOp/CppInterOp.h")

    // Neueres xeus-cpp / clang-repl Setup
    #include "CppInterOp/CppInterOp.h"

    Cpp::LoadLibrary("libopencv_core.so");
    Cpp::LoadLibrary("libopencv_imgproc.so");
    Cpp::LoadLibrary("libopencv_imgcodecs.so");
    Cpp::LoadLibrary("libopencv_video.so");
    Cpp::LoadLibrary("libopencv_calib3d.so");

    #define MVSR_XEUS_CPP 1

#else

    // Bisheriges xeus-cling Setup
    #pragma cling add_include_path("/usr/include/opencv4")
    #pragma cling add_library_path("/usr/lib/x86_64-linux-gnu")

    #pragma cling load("opencv_core")
    #pragma cling load("opencv_imgproc")
    #pragma cling load("opencv_imgcodecs")
    #pragma cling load("opencv_video")
    #pragma cling load("opencv_calib3d")

#endif

#include <opencv2/opencv.hpp>

std::cout << "OpenCV-Version: "
          << CV_VERSION
          << std::endl;

std::cout << "C++ Standard: "
          << __cplusplus
          << std::endl;

### Hilfsfunktion zur Bilddarstellung

OpenCV-Bilder werden direkt im Notebook ausgegeben.

In [ ]:
#include "nlohmann/json.hpp"

#ifdef MVSR_XEUS_CPP

    #include "xeus/xbase64.hpp"
    #include "xcpp/xdisplay.hpp"

#else

    #include "xtl/xbase64.hpp"

#endif

namespace nl = nlohmann;

namespace mvsr
{
    struct NotebookImage
    {
        std::string png_data;

        explicit NotebookImage(
            const cv::Mat& image
        )
        {
            std::vector<
                unsigned char
            > buffer;

            cv::imencode(
                ".png",
                image,
                buffer
            );

            png_data.assign(
                reinterpret_cast<
                    const char*
                >(buffer.data()),
                buffer.size()
            );
        }
    };

    nl::json mime_bundle_repr(
        const NotebookImage& image
    )
    {
        auto bundle =
            nl::json::object();

        #ifdef MVSR_XEUS_CPP

            bundle["image/png"] =
                xeus::base64encode(
                    image.png_data
                );

        #else

            bundle["image/png"] =
                xtl::base64encode(
                    image.png_data
                );

        #endif

        return bundle;
    }
}

#ifdef MVSR_XEUS_CPP

void show_image(
    const cv::Mat& image
)
{
    xcpp::display(
        mvsr::NotebookImage(
            image
        )
    );
}

#else

mvsr::NotebookImage show_image(
    const cv::Mat& image
)
{
    return mvsr::NotebookImage(
        image
    );
}

#endif

### Hilfsfunktion für Downloads

In [ ]:
bool download_file(
    const std::string& url,
    const std::string& output_file
)
{
    std::string command =
        "wget -q \""
        + url
        + "\" -O \""
        + output_file
        + "\"";

    int result =
        std::system(
            command.c_str()
        );

    if (result != 0)
    {
        std::cerr
            << "Download fehlgeschlagen: "
            << url
            << std::endl;

        return false;
    }

    return true;
}

### Hilfsfunktion: allgemeiner XY-Plot

Die Funktion wird später für Fehlerverläufe, Tiefeneinfluss und Parametervergleiche verwendet.

In [ ]:
cv::Mat render_xy_series_plot(
    const std::vector<
        std::vector<cv::Point2d>
    >& series,
    const std::vector<std::string>& labels,
    const std::string& title,
    const std::string& x_label,
    const std::string& y_label
)
{
    const int width = 820;
    const int height = 520;

    const int left = 85;
    const int right = 35;
    const int top = 55;
    const int bottom = 70;

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(
            255,
            255,
            255
        )
    );

    bool first_point = true;

    double min_x = 0.0;
    double max_x = 1.0;
    double min_y = 0.0;
    double max_y = 1.0;

    for (const auto& curve : series)
    {
        for (const auto& point : curve)
        {
            if (first_point)
            {
                min_x = max_x = point.x;
                min_y = max_y = point.y;
                first_point = false;
            }
            else
            {
                min_x =
                    std::min(
                        min_x,
                        point.x
                    );

                max_x =
                    std::max(
                        max_x,
                        point.x
                    );

                min_y =
                    std::min(
                        min_y,
                        point.y
                    );

                max_y =
                    std::max(
                        max_y,
                        point.y
                    );
            }
        }
    }

    double range_x =
        std::max(
            max_x - min_x,
            1e-9
        );

    double range_y =
        std::max(
            max_y - min_y,
            1e-9
        );

    min_x -= 0.05 * range_x;
    max_x += 0.05 * range_x;

    min_y -= 0.08 * range_y;
    max_y += 0.08 * range_y;

    range_x = max_x - min_x;
    range_y = max_y - min_y;

    auto to_pixel =
        [&](const cv::Point2d& point)
        {
            int x = cvRound(
                left
                + (
                    point.x - min_x
                )
                / range_x
                * (
                    width
                    - left
                    - right
                )
            );

            int y = cvRound(
                height
                - bottom
                - (
                    point.y - min_y
                )
                / range_y
                * (
                    height
                    - top
                    - bottom
                )
            );

            return cv::Point(
                x,
                y
            );
        };

    cv::line(
        canvas,
        cv::Point(
            left,
            top
        ),
        cv::Point(
            left,
            height - bottom
        ),
        cv::Scalar(
            0,
            0,
            0
        ),
        2
    );

    cv::line(
        canvas,
        cv::Point(
            left,
            height - bottom
        ),
        cv::Point(
            width - right,
            height - bottom
        ),
        cv::Scalar(
            0,
            0,
            0
        ),
        2
    );

    std::vector<cv::Scalar> colors = {
        cv::Scalar(220, 70, 40),
        cv::Scalar(50, 150, 40),
        cv::Scalar(40, 80, 220),
        cv::Scalar(170, 80, 170)
    };

    for (
        std::size_t curve_index = 0;
        curve_index < series.size();
        ++curve_index
    )
    {
        const auto& curve =
            series[curve_index];

        cv::Scalar color =
            colors[
                curve_index
                % colors.size()
            ];

        for (
            std::size_t i = 1;
            i < curve.size();
            ++i
        )
        {
            cv::line(
                canvas,
                to_pixel(
                    curve[i - 1]
                ),
                to_pixel(
                    curve[i]
                ),
                color,
                2,
                cv::LINE_AA
            );
        }
    }

    cv::putText(
        canvas,
        title,
        cv::Point(
            20,
            30
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.65,
        cv::Scalar(
            0,
            0,
            0
        ),
        2
    );

    cv::putText(
        canvas,
        x_label,
        cv::Point(
            width / 2 - 70,
            height - 20
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.5,
        cv::Scalar(
            0,
            0,
            0
        ),
        1
    );

    cv::putText(
        canvas,
        y_label,
        cv::Point(
            8,
            50
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.45,
        cv::Scalar(
            0,
            0,
            0
        ),
        1
    );

    for (
        std::size_t i = 0;
        i < labels.size()
            && i < series.size();
        ++i
    )
    {
        cv::Scalar color =
            colors[
                i
                % colors.size()
            ];

        int y =
            48
            + static_cast<int>(
                22 * i
            );

        cv::line(
            canvas,
            cv::Point(
                width - 210,
                y
            ),
            cv::Point(
                width - 180,
                y
            ),
            color,
            3
        );

        cv::putText(
            canvas,
            labels[i],
            cv::Point(
                width - 170,
                y + 5
            ),
            cv::FONT_HERSHEY_SIMPLEX,
            0.42,
            cv::Scalar(
                0,
                0,
                0
            ),
            1
        );
    }

    return canvas;
}

## 1. Von Wahrnehmung zu Regelung

Bisher:

**Kamerabild → Merkmale → Korrespondenzen → Pose**

Das ist noch keine geschlossene Regelung.

Eine einmalige Messung kann durch Kalibrierfehler, ungenaue Pose-Schätzung, Roboterpositionierfehler, Modellabweichungen oder Objektbewegung verfälscht werden.

Visual Servoing nutzt deshalb die Kamera kontinuierlich als Sensor:

**Kamera → visueller Fehler → Roboterbewegung → neues Bild → neuer visueller Fehler**

### Visueller Zustand und Regelfehler

Der aktuelle visuelle Zustand wird durch $\mathbf{s}$ beschrieben, der gewünschte Zustand durch $\mathbf{s}^{*}$.

Der Regelfehler lautet:

$\mathbf{e}=\mathbf{s}-\mathbf{s}^{*}$

Ziel:

$\mathbf{e}(t)\rightarrow\mathbf{0}$

Mögliche visuelle Merkmale sind beispielsweise Bildpunkte, Linien, Konturen oder eine Objektpose.

### Visual Servoing als Regelkreis

| Regelungstechnik | Visual Servoing |
|---|---|
| Führungsgröße | gewünschter visueller Zustand $\mathbf{s}^{*}$ |
| Regelgröße | aktueller visueller Zustand $\mathbf{s}(t)$ |
| Regelfehler | $\mathbf{e}(t)=\mathbf{s}(t)-\mathbf{s}^{*}$ |
| Regler | Visual-Servoing-Regler |
| Stellgröße | Kamerageschwindigkeit $\mathbf{v}_c$ |
| Strecke | Roboter + Kamera + Szene |
| Rückführung | Kamerabild / visuelle Merkmale |

Die grundlegende Struktur entspricht damit einem klassischen geschlossenen Regelkreis.

## 2. Bewegung im Bild – Optical Flow

Ein Bildpunkt liegt zum Zeitpunkt $t$ bei

$\mathbf{s}(t)=\begin{bmatrix}x(t)&y(t)\end{bmatrix}^{T}$

und kurze Zeit später an einer leicht veränderten Position.

Die Bildgeschwindigkeit lautet:

$\dot{\mathbf{s}}=\begin{bmatrix}\dot{x}&\dot{y}\end{bmatrix}^{T}$

Optical Flow beschreibt diese Bewegung von Bildinformation über die Zeit.

Für Visual Servoing ist aber vor allem folgende Frage interessant:

> **Welche Kamerabewegung erzeugt eine bestimmte Bildbewegung?**

### Beispiel: Optical Flow mit einem OpenCV-Testbild

Wir verwenden das OpenCV-Beispielbild `box.png` und detektieren darin markante Punkte mit **Shi-Tomasi**.

Anschließend erzeugen wir drei kontrollierte Bildänderungen:

- seitliche Verschiebung,
- Vergrößerung als einfache Näherung einer Kamerabewegung in **Z-Richtung**,
- Rotation um die optische Achse.

Die Features werden danach mit **Lucas-Kanade Optical Flow** verfolgt.

> Die Bildtransformationen sind bewusst vereinfacht. Bei einer echten 3D-Szene hängt die Bildbewegung zusätzlich von der Tiefe der einzelnen Punkte ab.

In [ ]:
std::system(
    "mkdir -p vs_data"
);

bool box_downloaded =
    download_file(
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/box.png",
        "vs_data/box.png"
    );

cv::Mat box_image =
    cv::imread(
        "vs_data/box.png",
        cv::IMREAD_GRAYSCALE
    );

std::vector<cv::Point2f>
    box_features;

cv::goodFeaturesToTrack(
    box_image,
    box_features,
    50,
    0.01,
    10.0
);

cv::Mat box_features_image;

cv::cvtColor(
    box_image,
    box_features_image,
    cv::COLOR_GRAY2BGR
);

for (const auto& point : box_features)
{
    cv::circle(
        box_features_image,
        point,
        3,
        cv::Scalar(
            0,
            0,
            255
        ),
        -1,
        cv::LINE_AA
    );
}

std::cout
    << "Detektierte Features: "
    << box_features.size()
    << std::endl;

In [ ]:
show_image(
    box_features_image
)

### Hilfsfunktion für eine kontrollierte Bildbewegung

In [ ]:
cv::Mat transform_vs_image(
    const cv::Mat& image,
    int motion_type
)
{
    cv::Mat matrix;

    cv::Point2f center(
        image.cols / 2.0F,
        image.rows / 2.0F
    );

    if (motion_type == 0)
    {
        matrix = (
            cv::Mat_<double>(
                2,
                3
            )
            <<
            1.0, 0.0, 30.0,
            0.0, 1.0, -12.0
        );
    }
    else if (motion_type == 1)
    {
        matrix =
            cv::getRotationMatrix2D(
                center,
                0.0,
                1.15
            );
    }
    else
    {
        matrix =
            cv::getRotationMatrix2D(
                center,
                12.0,
                1.0
            );
    }

    cv::Mat transformed;

    cv::warpAffine(
        image,
        transformed,
        matrix,
        image.size()
    );

    return transformed;
}

### Hilfsfunktion zur Darstellung des Optical Flow

In [ ]:
cv::Mat draw_optical_flow(
    const cv::Mat& image,
    const std::vector<cv::Point2f>& start_points,
    const std::vector<cv::Point2f>& end_points,
    const std::string& title
)
{
    cv::Mat visualization;

    cv::cvtColor(
        image,
        visualization,
        cv::COLOR_GRAY2BGR
    );

    std::size_t count =
        std::min(
            start_points.size(),
            end_points.size()
        );

    for (
        std::size_t i = 0;
        i < count;
        ++i
    )
    {
        cv::arrowedLine(
            visualization,
            start_points[i],
            end_points[i],
            cv::Scalar(
                0,
                0,
                255
            ),
            1,
            cv::LINE_AA,
            0,
            0.25
        );

        cv::circle(
            visualization,
            end_points[i],
            2,
            cv::Scalar(
                0,
                180,
                0
            ),
            -1,
            cv::LINE_AA
        );
    }

    cv::putText(
        visualization,
        title,
        cv::Point(
            10,
            24
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.6,
        cv::Scalar(
            255,
            80,
            20
        ),
        2
    );

    return visualization;
}

In [ ]:
std::vector<cv::Mat>
    optical_flow_images;

std::vector<std::string>
    optical_flow_titles = {
        "Seitliche Translation",
        "Bewegung in Z-Richtung",
        "Rotation"
    };

for (
    int motion_type = 0;
    motion_type < 3;
    ++motion_type
)
{
    cv::Mat transformed =
        transform_vs_image(
            box_image,
            motion_type
        );

    std::vector<cv::Point2f>
        tracked_points;

    std::vector<unsigned char>
        status;

    std::vector<float>
        error;

    cv::calcOpticalFlowPyrLK(
        box_image,
        transformed,
        box_features,
        tracked_points,
        status,
        error,
        cv::Size(
            21,
            21
        ),
        3
    );

    std::vector<cv::Point2f>
        valid_start;

    std::vector<cv::Point2f>
        valid_end;

    for (
        std::size_t i = 0;
        i < status.size();
        ++i
    )
    {
        if (status[i])
        {
            valid_start.push_back(
                box_features[i]
            );

            valid_end.push_back(
                tracked_points[i]
            );
        }
    }

    optical_flow_images.push_back(
        draw_optical_flow(
            transformed,
            valid_start,
            valid_end,
            optical_flow_titles[
                motion_type
            ]
        )
    );
}

cv::Mat optical_flow_overview;

cv::hconcat(
    optical_flow_images,
    optical_flow_overview
);

show_image(
    optical_flow_overview
)

### Beobachtung

Die Bewegungsmuster unterscheiden sich deutlich:

- **seitliche Translation:** Features bewegen sich überwiegend in eine gemeinsame Richtung,
- **Z-Bewegung:** Features bewegen sich radial vom bzw. zum Bildzentrum,
- **Rotation:** Features bewegen sich tangential um das Rotationszentrum.

Genau diese unterschiedlichen Bildbewegungen werden später durch die Spalten der **Interaction Matrix** beschrieben.

## 3. Vom Optical Flow zur Interaction Matrix

Für eine statische Szene wird die beobachtete Bildbewegung durch die Kamerabewegung verursacht.

Die Kamerageschwindigkeit besitzt sechs Freiheitsgrade:

$\mathbf{v}_c=
\begin{bmatrix}
v_x&v_y&v_z&\omega_x&\omega_y&\omega_z
\end{bmatrix}^{T}$

Der lokale Zusammenhang lautet:

$\dot{\mathbf{s}}=\mathbf{L}_s\mathbf{v}_c$

$\mathbf{L}_s$ heißt **Bild-Jacobi-Matrix**, **Image Jacobian** oder **Interaction Matrix**.

### Interaction Matrix eines Bildpunktes

Für normalisierte Bildkoordinaten $(x,y)$ und Tiefe $Z$ gilt:

$\mathbf{L}_p=
\begin{bmatrix}
-\frac{1}{Z} & 0 & \frac{x}{Z} & xy & -(1+x^2) & y \\
0 & -\frac{1}{Z} & \frac{y}{Z} & 1+y^2 & -xy & -x
\end{bmatrix}$

Ein Bildpunkt liefert zwei Messgrößen, während die Kamera sechs Bewegungsfreiheitsgrade besitzt.

Daher gilt:

$\mathbf{L}_p\in\mathbb{R}^{2\times6}$

In [ ]:
cv::Mat point_interaction_matrix(
    double x,
    double y,
    double Z
)
{
    return (
        cv::Mat_<double>(
            2,
            6
        )
        <<
        -1.0 / Z,
        0.0,
        x / Z,
        x * y,
        -(1.0 + x * x),
        y,

        0.0,
        -1.0 / Z,
        y / Z,
        1.0 + y * y,
        -x * y,
        -x
    );
}

In [ ]:
double x_point = 0.20;
double y_point = -0.15;
double Z_point = 1.50;

cv::Mat L_point =
    point_interaction_matrix(
        x_point,
        y_point,
        Z_point
    );

std::cout
    << L_point
    << std::endl;

std::cout
    << "Form: "
    << L_point.rows
    << " x "
    << L_point.cols
    << std::endl;

### Bedeutung der sechs Spalten

Jede Spalte der Interaction Matrix beschreibt den Einfluss eines Kamera-Freiheitsgrades auf die Bildbewegung.

Wir betrachten daher nacheinander:

$v_x,\;v_y,\;v_z,\;\omega_x,\;\omega_y,\;\omega_z$

### Hilfsfunktion zur Darstellung eines Flow-Feldes

In [ ]:
cv::Mat render_flow_field(
    const std::vector<cv::Point2d>& points,
    const std::vector<cv::Point2d>& flow,
    const std::string& title
)
{
    const int width = 420;
    const int height = 340;

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(
            255,
            255,
            255
        )
    );

    const double min_x = -0.7;
    const double max_x = 0.7;
    const double min_y = -0.6;
    const double max_y = 0.6;

    auto to_pixel =
        [&](const cv::Point2d& point)
        {
            int x = cvRound(
                25.0
                + (
                    point.x - min_x
                )
                / (
                    max_x - min_x
                )
                * (
                    width - 50
                )
            );

            int y = cvRound(
                height
                - 25.0
                - (
                    point.y - min_y
                )
                / (
                    max_y - min_y
                )
                * (
                    height - 70
                )
            );

            return cv::Point(
                x,
                y
            );
        };

    for (
        std::size_t i = 0;
        i < points.size();
        ++i
    )
    {
        cv::Point2d end =
            points[i]
            + flow[i] * 4.0;

        cv::arrowedLine(
            canvas,
            to_pixel(
                points[i]
            ),
            to_pixel(
                end
            ),
            cv::Scalar(
                30,
                90,
                220
            ),
            1,
            cv::LINE_AA,
            0,
            0.25
        );

        cv::circle(
            canvas,
            to_pixel(
                points[i]
            ),
            2,
            cv::Scalar(
                20,
                20,
                20
            ),
            -1
        );
    }

    cv::putText(
        canvas,
        title,
        cv::Point(
            10,
            22
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.55,
        cv::Scalar(
            0,
            0,
            0
        ),
        1
    );

    return canvas;
}

In [ ]:
std::vector<cv::Point2d>
    grid_points;

for (
    double y = -0.4;
    y <= 0.4001;
    y += 0.2
)
{
    for (
        double x = -0.5;
        x <= 0.5001;
        x += 1.0 / 6.0
    )
    {
        grid_points.emplace_back(
            x,
            y
        );
    }
}

std::vector<std::string>
    motion_names = {
        "Translation vx",
        "Translation vy",
        "Translation vz",
        "Rotation wx",
        "Rotation wy",
        "Rotation wz"
    };

std::vector<cv::Mat>
    flow_panels;

for (
    int dof = 0;
    dof < 6;
    ++dof
)
{
    cv::Mat camera_motion =
        cv::Mat::zeros(
            6,
            1,
            CV_64F
        );

    camera_motion.at<double>(
        dof,
        0
    ) = 0.10;

    std::vector<cv::Point2d>
        flow;

    for (const auto& point : grid_points)
    {
        cv::Mat L =
            point_interaction_matrix(
                point.x,
                point.y,
                1.5
            );

        cv::Mat velocity =
            L * camera_motion;

        flow.emplace_back(
            velocity.at<double>(
                0,
                0
            ),
            velocity.at<double>(
                1,
                0
            )
        );
    }

    flow_panels.push_back(
        render_flow_field(
            grid_points,
            flow,
            motion_names[dof]
        )
    );
}

cv::Mat flow_row_1;
cv::Mat flow_row_2;
cv::Mat flow_overview;

std::vector<cv::Mat>
    first_row = {
        flow_panels[0],
        flow_panels[1],
        flow_panels[2]
    };

std::vector<cv::Mat>
    second_row = {
        flow_panels[3],
        flow_panels[4],
        flow_panels[5]
    };

cv::hconcat(
    first_row,
    flow_row_1
);

cv::hconcat(
    second_row,
    flow_row_2
);

cv::vconcat(
    flow_row_1,
    flow_row_2,
    flow_overview
);

show_image(
    flow_overview
)

### Beobachtung

Die sechs Freiheitsgrade erzeugen charakteristische Flussmuster:

- $v_x$: überwiegend horizontale Verschiebung,
- $v_y$: überwiegend vertikale Verschiebung,
- $v_z$: Expansion bzw. Kontraktion,
- Rotationen: positionsabhängige Flussmuster.

Damit bekommt die Matrix eine direkte geometrische Bedeutung:

**Kamerabewegung → Bildbewegung**

### Mini-Aufgabe: Kamerabewegung

Verändern Sie die sechs Komponenten von `v_test`.

Beobachten Sie, wie sich Translationen und Rotationen auf die Bewegung eines einzelnen Bildpunktes auswirken.

In [ ]:
// TODO: Kamerageschwindigkeit verändern

cv::Mat v_test = (
    cv::Mat_<double>(
        6,
        1
    )
    <<
    0.05,   // vx
    0.00,   // vy
    0.00,   // vz
    0.00,   // wx
    0.05,   // wy
    0.00    // wz
);

cv::Mat flow_test =
    L_point
    * v_test;

std::cout
    << "Bildgeschwindigkeit:"
    << std::endl
    << flow_test
    << std::endl;

## 4. Einfluss der Tiefe

Die translatorischen Terme der Interaction Matrix enthalten $1/Z$.

Daher erzeugt dieselbe translatorische Kamerageschwindigkeit bei einem nahen Objekt eine stärkere Bildbewegung als bei einem weit entfernten Objekt.

In [ ]:
std::vector<cv::Point2d>
    depth_curve;

cv::Mat depth_motion = (
    cv::Mat_<double>(
        6,
        1
    )
    <<
    0.10,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0
);

for (
    int i = 0;
    i < 100;
    ++i
)
{
    double Z =
        0.4
        + i
        * (
            4.0 - 0.4
        )
        / 99.0;

    cv::Mat L =
        point_interaction_matrix(
            x_point,
            y_point,
            Z
        );

    cv::Mat velocity =
        L
        * depth_motion;

    double magnitude =
        cv::norm(
            velocity
        );

    depth_curve.emplace_back(
        Z,
        magnitude
    );
}

std::vector<
    std::vector<cv::Point2d>
> depth_series = {
    depth_curve
};

std::vector<std::string>
    depth_labels = {
        "Bildgeschwindigkeit"
    };

cv::Mat depth_plot =
    render_xy_series_plot(
        depth_series,
        depth_labels,
        "Einfluss der Tiefe",
        "Tiefe Z",
        "Betrag der Bildgeschwindigkeit"
    );

show_image(
    depth_plot
)

## 5. Mehrere Bildpunkte

Ein einzelner Bildpunkt liefert nur zwei Gleichungen.

Für mehrere Features werden die einzelnen Interaction Matrices vertikal gestapelt:

$\mathbf{L}_s=
\begin{bmatrix}
\mathbf{L}_{p_1}\\
\mathbf{L}_{p_2}\\
\vdots\\
\mathbf{L}_{p_N}
\end{bmatrix}$

Damit gilt weiterhin:

$\dot{\mathbf{s}}=\mathbf{L}_s\mathbf{v}_c$

In [ ]:
cv::Mat stacked_interaction_matrix(
    const std::vector<cv::Point2d>& points,
    const std::vector<double>& depths
)
{
    cv::Mat L =
        cv::Mat::zeros(
            static_cast<int>(
                2 * points.size()
            ),
            6,
            CV_64F
        );

    for (
        std::size_t i = 0;
        i < points.size();
        ++i
    )
    {
        cv::Mat L_point =
            point_interaction_matrix(
                points[i].x,
                points[i].y,
                depths[i]
            );

        L_point.copyTo(
            L.rowRange(
                static_cast<int>(
                    2 * i
                ),
                static_cast<int>(
                    2 * i + 2
                )
            )
        );
    }

    return L;
}

### Hilfsfunktion für den numerischen Rang

In [ ]:
int matrix_rank(
    const cv::Mat& matrix,
    double relative_tolerance = 1e-9
)
{
    cv::SVD svd(
        matrix,
        cv::SVD::NO_UV
    );

    double maximum =
        0.0;

    for (
        int i = 0;
        i < svd.w.rows;
        ++i
    )
    {
        maximum =
            std::max(
                maximum,
                svd.w.at<double>(
                    i,
                    0
                )
            );
    }

    double threshold =
        relative_tolerance
        * std::max(
            matrix.rows,
            matrix.cols
        )
        * maximum;

    int rank = 0;

    for (
        int i = 0;
        i < svd.w.rows;
        ++i
    )
    {
        if (
            svd.w.at<double>(
                i,
                0
            )
            > threshold
        )
        {
            ++rank;
        }
    }

    return rank;
}

In [ ]:
std::vector<cv::Point2d>
    feature_points = {
        {-0.25, -0.20},
        { 0.25, -0.20},
        { 0.25,  0.20},
        {-0.25,  0.20}
    };

std::vector<double>
    feature_depths(
        feature_points.size(),
        1.5
    );

cv::Mat L_features =
    stacked_interaction_matrix(
        feature_points,
        feature_depths
    );

std::cout
    << "Form: "
    << L_features.rows
    << " x "
    << L_features.cols
    << std::endl;

std::cout
    << "Rang: "
    << matrix_rank(
        L_features
    )
    << std::endl;

### Rang und Anzahl der Features

Der gestapelte Image Jacobian muss genügend unabhängige Information enthalten, damit die gewünschten Freiheitsgrade kontrolliert werden können.

Ein Punkt liefert maximal Rang 2, zwei Punkte maximal Rang 4.

Ab drei geometrisch günstig angeordneten Punkten kann theoretisch Rang 6 erreicht werden.

In [ ]:
for (
    int number_of_points = 1;
    number_of_points <= 4;
    ++number_of_points
)
{
    std::vector<cv::Point2d>
        points_subset(
            feature_points.begin(),
            feature_points.begin()
                + number_of_points
        );

    std::vector<double>
        depths_subset(
            number_of_points,
            1.5
        );

    cv::Mat L_subset =
        stacked_interaction_matrix(
            points_subset,
            depths_subset
        );

    std::cout
        << number_of_points
        << " Punkt(e): "
        << "Form = "
        << L_subset.rows
        << " x "
        << L_subset.cols
        << ", Rang = "
        << matrix_rank(
            L_subset
        )
        << std::endl;
}

## 6. Regelungstechnische Interpretation

Lokal gilt:

$\dot{\mathbf{s}}=\mathbf{L}_s\mathbf{v}_c$

Bei lokal konstantem $\mathbf{L}_s$ folgt im Laplace-Bereich:

$\mathbf{S}(s)=\frac{1}{s}\mathbf{L}_s\mathbf{V}_c(s)$

Damit ergibt sich die lokale Strecke:

$\mathbf{G}_S(s)=\frac{1}{s}\mathbf{L}_s$

Die Strecke enthält also

- die geometrische Abbildung $\mathbf{L}_s$,
- einen Integrator von Merkmalsgeschwindigkeit zu Merkmalsposition.

### P-Regler an einer I-Strecke

Für einen skalaren Fehler gilt vereinfacht:

$\dot e = u$

Mit einem P-Regler

$u=-\lambda e$

folgt:

$\dot e=-\lambda e$

Der Fehler fällt damit exponentiell ab.

Diese Struktur taucht im klassischen IBVS wieder auf.

In [ ]:
std::vector<cv::Point2d>
simulate_scalar_error(
    double initial_error,
    double gain,
    double dt,
    double duration
)
{
    std::vector<cv::Point2d>
        history;

    double error =
        initial_error;

    int steps =
        static_cast<int>(
            duration / dt
        );

    for (
        int step = 0;
        step <= steps;
        ++step
    )
    {
        double time =
            step * dt;

        history.emplace_back(
            time,
            error
        );

        double control =
            -gain
            * error;

        error +=
            dt
            * control;
    }

    return history;
}

In [ ]:
std::vector<cv::Point2d>
    scalar_error =
        simulate_scalar_error(
            1.0,
            1.0,
            0.01,
            5.0
        );

std::vector<
    std::vector<cv::Point2d>
> scalar_series = {
    scalar_error
};

std::vector<std::string>
    scalar_labels = {
        "lambda = 1.0"
    };

cv::Mat scalar_plot =
    render_xy_series_plot(
        scalar_series,
        scalar_labels,
        "P-Regler an einer I-Strecke",
        "Zeit",
        "Fehler e"
    );

show_image(
    scalar_plot
)

### Mini-Aufgabe: Verstärkung

Verändern Sie `gain_test`.

Beobachten Sie, wie sich die Geschwindigkeit des Fehlerabbaus verändert.

In [ ]:
// TODO: Verstärkung verändern

double gain_test = 2.0;

std::vector<cv::Point2d>
    error_test =
        simulate_scalar_error(
            1.0,
            gain_test,
            0.01,
            5.0
        );

std::vector<
    std::vector<cv::Point2d>
> gain_test_series = {
    error_test
};

std::vector<std::string>
    gain_test_labels = {
        "gewählte Verstärkung"
    };

cv::Mat gain_test_plot =
    render_xy_series_plot(
        gain_test_series,
        gain_test_labels,
        "P-Regler: Einfluss von lambda",
        "Zeit",
        "Fehler e"
    );

show_image(
    gain_test_plot
)

## 7. Herleitung des IBVS-Regelgesetzes

Der Fehler lautet:

$\mathbf{e}=\mathbf{s}-\mathbf{s}^{*}$

Für ein konstantes Ziel gilt:

$\dot{\mathbf{e}}=\dot{\mathbf{s}}$

Gewünschtes Fehlerverhalten:

$\dot{\mathbf{e}}=-\lambda\mathbf{e}$

Mit

$\dot{\mathbf{s}}=\mathbf{L}_s\mathbf{v}_c$

folgt:

$\mathbf{L}_s\mathbf{v}_c=-\lambda\mathbf{e}$

und schließlich:

$\boxed{\mathbf{v}_c=-\lambda\mathbf{L}_s^{+}\mathbf{e}}$

### Bedeutung

$\mathbf{L}_s$ beantwortet:

**Kamerabewegung → Bildbewegung**

Die Pseudoinverse $\mathbf{L}_s^{+}$ wird im Regler verwendet, um näherungsweise die umgekehrte Frage zu beantworten:

**gewünschte Bildbewegung → Kamerabewegung**

Lokal kann IBVS damit als **P-Regelung einer I-Strecke mit geometrischer Transformation** interpretiert werden.

## 8. IBVS – Regelung direkt im Bildraum

Bei IBVS werden Bildmerkmale direkt als Regelgrößen verwendet:

$\mathbf{s}=
\begin{bmatrix}
x_1&y_1&x_2&y_2&\cdots
\end{bmatrix}^{T}$

$\mathbf{s}$ beschreibt die aktuellen Bildfeatures, $\mathbf{s}^{*}$ die korrespondierenden Features im Zielbild.

In einer realen Anwendung müssen diese Korrespondenzen beispielsweise durch Feature Matching bestimmt werden.

Für die folgende Simulation sind die Korrespondenzen bereits bekannt.

### Hilfsfunktionen für eine 3D-Kamerasimulation

In [ ]:
cv::Matx33d exp_so3(
    const cv::Vec3d& rotation_vector
)
{
    cv::Mat rotation_matrix;

    cv::Rodrigues(
        rotation_vector,
        rotation_matrix
    );

    cv::Matx33d result;

    for (
        int r = 0;
        r < 3;
        ++r
    )
    {
        for (
            int c = 0;
            c < 3;
            ++c
        )
        {
            result(
                r,
                c
            ) =
                rotation_matrix.at<double>(
                    r,
                    c
                );
        }
    }

    return result;
}

In [ ]:
cv::Vec3d log_so3(
    const cv::Matx33d& rotation_matrix
)
{
    cv::Mat matrix(
        3,
        3,
        CV_64F
    );

    for (
        int r = 0;
        r < 3;
        ++r
    )
    {
        for (
            int c = 0;
            c < 3;
            ++c
        )
        {
            matrix.at<double>(
                r,
                c
            ) =
                rotation_matrix(
                    r,
                    c
                );
        }
    }

    cv::Mat rotation_vector;

    cv::Rodrigues(
        matrix,
        rotation_vector
    );

    return cv::Vec3d(
        rotation_vector.at<double>(
            0,
            0
        ),
        rotation_vector.at<double>(
            1,
            0
        ),
        rotation_vector.at<double>(
            2,
            0
        )
    );
}

### Projektion der 3D-Punkte

In [ ]:
void project_points_normalized(
    const std::vector<cv::Point3d>& points_world,
    const cv::Matx33d& R_wc,
    const cv::Vec3d& t_wc,
    std::vector<cv::Point2d>& features,
    std::vector<cv::Point3d>& points_camera
)
{
    features.clear();
    points_camera.clear();

    cv::Matx33d R_cw =
        R_wc.t();

    for (const auto& point : points_world)
    {
        cv::Vec3d p_world(
            point.x,
            point.y,
            point.z
        );

        cv::Vec3d p_camera =
            R_cw
            * (
                p_world
                - t_wc
            );

        points_camera.emplace_back(
            p_camera[0],
            p_camera[1],
            p_camera[2]
        );

        features.emplace_back(
            p_camera[0]
                / p_camera[2],
            p_camera[1]
                / p_camera[2]
        );
    }
}

### Aktualisierung der Kamerapose

In [ ]:
void integrate_camera_pose(
    cv::Matx33d& R_wc,
    cv::Vec3d& t_wc,
    const cv::Mat& v_c,
    double dt
)
{
    cv::Vec3d translation_velocity(
        v_c.at<double>(
            0,
            0
        ),
        v_c.at<double>(
            1,
            0
        ),
        v_c.at<double>(
            2,
            0
        )
    );

    cv::Vec3d rotation_velocity(
        v_c.at<double>(
            3,
            0
        ),
        v_c.at<double>(
            4,
            0
        ),
        v_c.at<double>(
            5,
            0
        )
    );

    t_wc =
        t_wc
        + R_wc
        * (
            translation_velocity
            * dt
        );

    R_wc =
        R_wc
        * exp_so3(
            rotation_velocity
            * dt
        );
}

### Hilfsfunktion: Feature-Trajektorien

In [ ]:
cv::Mat render_feature_trajectories(
    const std::vector<
        std::vector<cv::Point2d>
    >& feature_history,
    const std::vector<cv::Point2d>& target_features,
    const std::string& title
)
{
    const int width = 700;
    const int height = 600;

    const int left = 45;
    const int right = 35;
    const int top = 45;
    const int bottom = 45;

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(
            255,
            255,
            255
        )
    );

    if (
        feature_history.empty()
        || target_features.empty()
    )
    {
        return canvas;
    }

    double min_x =
        target_features.front().x;

    double max_x =
        target_features.front().x;

    double min_y =
        target_features.front().y;

    double max_y =
        target_features.front().y;

    for (const auto& frame : feature_history)
    {
        for (const auto& point : frame)
        {
            min_x =
                std::min(
                    min_x,
                    point.x
                );

            max_x =
                std::max(
                    max_x,
                    point.x
                );

            min_y =
                std::min(
                    min_y,
                    point.y
                );

            max_y =
                std::max(
                    max_y,
                    point.y
                );
        }
    }

    for (const auto& point : target_features)
    {
        min_x =
            std::min(
                min_x,
                point.x
            );

        max_x =
            std::max(
                max_x,
                point.x
            );

        min_y =
            std::min(
                min_y,
                point.y
            );

        max_y =
            std::max(
                max_y,
                point.y
            );
    }

    double range_x =
        std::max(
            max_x - min_x,
            1e-6
        );

    double range_y =
        std::max(
            max_y - min_y,
            1e-6
        );

    min_x -=
        0.10
        * range_x;

    max_x +=
        0.10
        * range_x;

    min_y -=
        0.10
        * range_y;

    max_y +=
        0.10
        * range_y;

    range_x =
        max_x - min_x;

    range_y =
        max_y - min_y;

    auto to_pixel =
        [&](const cv::Point2d& point)
        {
            int x = cvRound(
                left
                + (
                    point.x - min_x
                )
                / range_x
                * (
                    width
                    - left
                    - right
                )
            );

            int y = cvRound(
                top
                + (
                    point.y - min_y
                )
                / range_y
                * (
                    height
                    - top
                    - bottom
                )
            );

            return cv::Point(
                x,
                y
            );
        };

    std::size_t feature_count =
        target_features.size();

    for (
        std::size_t feature_index = 0;
        feature_index < feature_count;
        ++feature_index
    )
    {
        for (
            std::size_t step = 1;
            step < feature_history.size();
            ++step
        )
        {
            cv::line(
                canvas,
                to_pixel(
                    feature_history[
                        step - 1
                    ][
                        feature_index
                    ]
                ),
                to_pixel(
                    feature_history[
                        step
                    ][
                        feature_index
                    ]
                ),
                cv::Scalar(
                    180,
                    90,
                    30
                ),
                2,
                cv::LINE_AA
            );
        }

        cv::circle(
            canvas,
            to_pixel(
                feature_history.front()[
                    feature_index
                ]
            ),
            5,
            cv::Scalar(
                30,
                120,
                220
            ),
            -1
        );

        cv::drawMarker(
            canvas,
            to_pixel(
                target_features[
                    feature_index
                ]
            ),
            cv::Scalar(
                30,
                170,
                30
            ),
            cv::MARKER_TILTED_CROSS,
            12,
            2
        );
    }

    cv::putText(
        canvas,
        title,
        cv::Point(
            20,
            28
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.65,
        cv::Scalar(
            0,
            0,
            0
        ),
        2
    );

    cv::putText(
        canvas,
        "Kreis = Start, X = Ziel",
        cv::Point(
            20,
            height - 14
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.45,
        cv::Scalar(
            0,
            0,
            0
        ),
        1
    );

    return canvas;
}

### Hilfsfunktion: Kameratrajektorie

Für die Darstellung der 3D-Trajektorie wird eine einfache isometrische Projektion verwendet.

In [ ]:
cv::Point2d project_isometric(
    const cv::Vec3d& point
)
{
    return cv::Point2d(
        point[0]
            + 0.55 * point[1],
        point[2]
            - 0.35 * point[1]
    );
}

In [ ]:
cv::Mat render_camera_trajectories(
    const std::vector<
        std::vector<cv::Vec3d>
    >& trajectories,
    const std::vector<std::string>& labels,
    const cv::Vec3d& start,
    const cv::Vec3d& target,
    const std::string& title
)
{
    std::vector<
        std::vector<cv::Point2d>
    > projected;

    for (const auto& trajectory : trajectories)
    {
        std::vector<cv::Point2d>
            curve;

        for (const auto& point : trajectory)
        {
            curve.push_back(
                project_isometric(
                    point
                )
            );
        }

        projected.push_back(
            curve
        );
    }

    cv::Mat plot =
        render_xy_series_plot(
            projected,
            labels,
            title,
            "isometrische x/y-Richtung",
            "isometrische z-Richtung"
        );

    return plot;
}

### Szene und Zielpose

Die acht 3D-Punkte bilden die Ecken eines kleinen Quaders.

Die Startpose ist bewusst deutlich von der Zielpose entfernt:

- Rotationsabweichung,
- seitlicher Versatz,
- Differenz in Z-Richtung.

Dadurch werden die unterschiedlichen Eigenschaften von IBVS und PBVS sichtbarer:

- **IBVS:** versucht primär, die Features im Bildraum zum Ziel zu führen,
- **PBVS:** reduziert direkt den Posefehler und folgt dadurch einer einfacheren 3D-Trajektorie.

In [ ]:
const double degree_to_rad =
    CV_PI / 180.0;

std::vector<cv::Point3d>
    points_world = {
        {-0.20, -0.15, 1.40},
        { 0.20, -0.15, 1.40},
        { 0.20,  0.15, 1.40},
        {-0.20,  0.15, 1.40},
        {-0.20, -0.15, 1.70},
        { 0.20, -0.15, 1.70},
        { 0.20,  0.15, 1.70},
        {-0.20,  0.15, 1.70}
    };

cv::Matx33d R_start =
    exp_so3(
        cv::Vec3d(
            35.0 * degree_to_rad,
            -35.0 * degree_to_rad,
            40.0 * degree_to_rad
        )
    );

cv::Vec3d t_start(
    -0.35,
     0.25,
    -0.80
);

cv::Matx33d R_target =
    exp_so3(
        cv::Vec3d(
            6.0 * degree_to_rad,
            -8.0 * degree_to_rad,
            5.0 * degree_to_rad
        )
    );

cv::Vec3d t_target(
    0.12,
    -0.08,
    0.10
);

std::vector<cv::Point2d>
    features_start;

std::vector<cv::Point3d>
    points_camera_start;

project_points_normalized(
    points_world,
    R_start,
    t_start,
    features_start,
    points_camera_start
);

std::vector<cv::Point2d>
    features_target;

std::vector<cv::Point3d>
    points_camera_target;

project_points_normalized(
    points_world,
    R_target,
    t_target,
    features_target,
    points_camera_target
);

### Start- und Sollfeatures

In [ ]:
std::vector<
    std::vector<cv::Point2d>
> start_target_history = {
    features_start,
    features_target
};

cv::Mat start_target_plot =
    render_feature_trajectories(
        start_target_history,
        features_target,
        "Start- und Sollfeatures"
    );

show_image(
    start_target_plot
)

### Geschlossener IBVS-Regelkreis

In jeder Iteration wird

1. der aktuelle Bildzustand bestimmt,
2. der visuelle Fehler berechnet,
3. die aktuelle Interaction Matrix aufgebaut,
4. die Kamerageschwindigkeit bestimmt,
5. die Kamerapose aktualisiert,
6. erneut gemessen.

### Datenstruktur für Simulationsergebnisse

In [ ]:
struct VSSimulationResult
{
    std::vector<double>
        error_history;

    std::vector<cv::Vec3d>
        camera_history;

    std::vector<
        std::vector<cv::Point2d>
    > feature_history;

    cv::Matx33d R_final;

    cv::Vec3d t_final;
};

In [ ]:
VSSimulationResult simulate_ibvs(
    const std::vector<cv::Point3d>& points_world,
    const cv::Matx33d& R_initial,
    const cv::Vec3d& t_initial,
    const cv::Matx33d& R_target,
    const cv::Vec3d& t_target,
    double gain,
    double dt,
    int iterations,
    double depth_scale
)
{
    std::vector<cv::Point2d>
        target_features;

    std::vector<cv::Point3d>
        target_camera_points;

    project_points_normalized(
        points_world,
        R_target,
        t_target,
        target_features,
        target_camera_points
    );

    cv::Matx33d R_wc =
        R_initial;

    cv::Vec3d t_wc =
        t_initial;

    VSSimulationResult result;

    for (
        int step = 0;
        step < iterations;
        ++step
    )
    {
        std::vector<cv::Point2d>
            features;

        std::vector<cv::Point3d>
            camera_points;

        project_points_normalized(
            points_world,
            R_wc,
            t_wc,
            features,
            camera_points
        );

        cv::Mat error(
            static_cast<int>(
                2 * features.size()
            ),
            1,
            CV_64F
        );

        std::vector<double>
            estimated_depths;

        for (
            std::size_t i = 0;
            i < features.size();
            ++i
        )
        {
            error.at<double>(
                static_cast<int>(
                    2 * i
                ),
                0
            ) =
                features[i].x
                - target_features[i].x;

            error.at<double>(
                static_cast<int>(
                    2 * i + 1
                ),
                0
            ) =
                features[i].y
                - target_features[i].y;

            estimated_depths.push_back(
                camera_points[i].z
                * depth_scale
            );
        }

        cv::Mat L =
            stacked_interaction_matrix(
                features,
                estimated_depths
            );

        cv::Mat L_pinv;

        cv::invert(
            L,
            L_pinv,
            cv::DECOMP_SVD
        );

        cv::Mat v_c =
            -gain
            * L_pinv
            * error;

        for (
            int i = 0;
            i < 3;
            ++i
        )
        {
            v_c.at<double>(
                i,
                0
            ) =
                std::max(
                    -0.25,
                    std::min(
                        0.25,
                        v_c.at<double>(
                            i,
                            0
                        )
                    )
                );
        }

        for (
            int i = 3;
            i < 6;
            ++i
        )
        {
            v_c.at<double>(
                i,
                0
            ) =
                std::max(
                    -0.8,
                    std::min(
                        0.8,
                        v_c.at<double>(
                            i,
                            0
                        )
                    )
                );
        }

        result.error_history.push_back(
            cv::norm(
                error
            )
        );

        result.camera_history.push_back(
            t_wc
        );

        result.feature_history.push_back(
            features
        );

        integrate_camera_pose(
            R_wc,
            t_wc,
            v_c,
            dt
        );
    }

    result.R_final =
        R_wc;

    result.t_final =
        t_wc;

    return result;
}

In [ ]:
VSSimulationResult ibvs_result =
    simulate_ibvs(
        points_world,
        R_start,
        t_start,
        R_target,
        t_target,
        0.8,
        0.05,
        300,
        1.0
    );

std::cout
    << "Startfehler: "
    << ibvs_result.error_history.front()
    << std::endl;

std::cout
    << "Endfehler: "
    << ibvs_result.error_history.back()
    << std::endl;

std::cout
    << "Zielposition: "
    << t_target
    << std::endl;

std::cout
    << "Erreichte Position: "
    << ibvs_result.t_final
    << std::endl;

### Fehlerverlauf

In [ ]:
std::vector<cv::Point2d>
    ibvs_error_curve;

for (
    std::size_t i = 0;
    i < ibvs_result.error_history.size();
    ++i
)
{
    ibvs_error_curve.emplace_back(
        static_cast<double>(
            i
        ),
        ibvs_result.error_history[i]
    );
}

std::vector<
    std::vector<cv::Point2d>
> ibvs_error_series = {
    ibvs_error_curve
};

std::vector<std::string>
    ibvs_error_labels = {
        "IBVS"
    };

cv::Mat ibvs_error_plot =
    render_xy_series_plot(
        ibvs_error_series,
        ibvs_error_labels,
        "IBVS: Fehlerverlauf",
        "Iteration",
        "||e||"
    );

show_image(
    ibvs_error_plot
)

### Feature-Trajektorien bei IBVS

IBVS reduziert den Fehler direkt im Bildraum.

Daher werden die Feature-Trajektorien unmittelbar durch die Regelung beeinflusst.

In [ ]:
cv::Mat ibvs_feature_plot =
    render_feature_trajectories(
        ibvs_result.feature_history,
        features_target,
        "IBVS: Feature-Trajektorien"
    );

show_image(
    ibvs_feature_plot
)

### Kameratrajektorie bei IBVS

Die Kameratrajektorie im 3D-Raum wird bei IBVS nicht direkt vorgegeben.

Sie ergibt sich indirekt aus der Reduktion des Bildfehlers.

In [ ]:
std::vector<
    std::vector<cv::Vec3d>
> ibvs_camera_series = {
    ibvs_result.camera_history
};

std::vector<std::string>
    ibvs_camera_labels = {
        "IBVS"
    };

cv::Mat ibvs_camera_plot =
    render_camera_trajectories(
        ibvs_camera_series,
        ibvs_camera_labels,
        t_start,
        t_target,
        "IBVS: Kameratrajektorie"
    );

show_image(
    ibvs_camera_plot
)

## 9. Einfluss der Reglerverstärkung

Im idealen lokalen Fall gilt:

$\dot{\mathbf{e}}\approx-\lambda\mathbf{e}$

Ein größeres $\lambda$ führt daher grundsätzlich zu einem schnelleren Fehlerabbau.

In realen Systemen können sehr große Werte jedoch zu hohen Stellgrößen, Sättigung oder Problemen durch Verzögerungen führen.

In [ ]:
std::vector<double>
    gains = {
        0.3,
        0.8,
        1.5
    };

std::vector<
    std::vector<cv::Point2d>
> gain_curves;

std::vector<std::string>
    gain_labels;

for (double gain : gains)
{
    VSSimulationResult result =
        simulate_ibvs(
            points_world,
            R_start,
            t_start,
            R_target,
            t_target,
            gain,
            0.05,
            300,
            1.0
        );

    std::vector<cv::Point2d>
        curve;

    for (
        std::size_t i = 0;
        i < result.error_history.size();
        ++i
    )
    {
        curve.emplace_back(
            static_cast<double>(
                i
            ),
            result.error_history[i]
        );
    }

    gain_curves.push_back(
        curve
    );

    gain_labels.push_back(
        "lambda = "
        + std::to_string(
            gain
        )
    );
}

cv::Mat gain_plot =
    render_xy_series_plot(
        gain_curves,
        gain_labels,
        "Einfluss der Reglerverstaerkung",
        "Iteration",
        "||e||"
    );

show_image(
    gain_plot
)

## 10. Falsche Tiefenschätzung

Die translatorischen Terme der Interaction Matrix benötigen die Tiefe $Z$.

In realen Anwendungen ist diese Tiefe oft nur näherungsweise bekannt.

Wir verwenden deshalb im Regler bewusst eine skalierte Tiefenschätzung.

In [ ]:
std::vector<double>
    depth_scales = {
        0.6,
        1.0,
        1.5
    };

std::vector<
    std::vector<cv::Point2d>
> depth_error_curves;

std::vector<std::string>
    depth_error_labels;

for (double scale : depth_scales)
{
    VSSimulationResult result =
        simulate_ibvs(
            points_world,
            R_start,
            t_start,
            R_target,
            t_target,
            0.8,
            0.05,
            300,
            scale
        );

    std::vector<cv::Point2d>
        curve;

    for (
        std::size_t i = 0;
        i < result.error_history.size();
        ++i
    )
    {
        curve.emplace_back(
            static_cast<double>(
                i
            ),
            result.error_history[i]
        );
    }

    depth_error_curves.push_back(
        curve
    );

    depth_error_labels.push_back(
        "Z-Faktor = "
        + std::to_string(
            scale
        )
    );
}

cv::Mat depth_error_plot =
    render_xy_series_plot(
        depth_error_curves,
        depth_error_labels,
        "Einfluss der Tiefenschaetzung",
        "Iteration",
        "||e||"
    );

show_image(
    depth_error_plot
)

### Beobachtung

IBVS kann auch bei einer ungenauen Tiefenschätzung konvergieren.

Eine falsche Tiefe verändert jedoch den transienten Verlauf und die berechnete Kamerageschwindigkeit.

Das zeigt, warum IBVS in der Praxis häufig relativ tolerant gegenüber moderaten Modellfehlern ist.

## 11. Position-Based Visual Servoing (PBVS)

Bei PBVS wird zunächst eine Pose geschätzt.

Die Regelgröße ist ein Posefehler:

$\mathbf{e}_p=
\begin{bmatrix}
\mathbf{t}_e\\
\boldsymbol{\theta}_e
\end{bmatrix}$

PBVS regelt damit direkt im **Pose-/3D-Raum**.

Für den Vergleich verwenden wir eine vereinfachte Pose-Regelung, bei der Translation und Rotation direkt reduziert werden.

In [ ]:
VSSimulationResult simulate_pbvs(
    const std::vector<cv::Point3d>& points_world,
    const cv::Matx33d& R_initial,
    const cv::Vec3d& t_initial,
    const cv::Matx33d& R_target,
    const cv::Vec3d& t_target,
    double gain,
    double dt,
    int iterations
)
{
    cv::Matx33d R_wc =
        R_initial;

    cv::Vec3d t_wc =
        t_initial;

    VSSimulationResult result;

    for (
        int step = 0;
        step < iterations;
        ++step
    )
    {
        cv::Vec3d translation_error_world =
            t_target
            - t_wc;

        cv::Matx33d R_error =
            R_wc.t()
            * R_target;

        cv::Vec3d rotation_error_camera =
            log_so3(
                R_error
            );

        cv::Vec3d translation_command_camera =
            R_wc.t()
            * (
                translation_error_world
                * gain
            );

        cv::Vec3d rotation_command_camera =
            rotation_error_camera
            * gain;

        cv::Mat v_c = (
            cv::Mat_<double>(
                6,
                1
            )
            <<
            translation_command_camera[0],
            translation_command_camera[1],
            translation_command_camera[2],
            rotation_command_camera[0],
            rotation_command_camera[1],
            rotation_command_camera[2]
        );

        std::vector<cv::Point2d>
            features;

        std::vector<cv::Point3d>
            camera_points;

        project_points_normalized(
            points_world,
            R_wc,
            t_wc,
            features,
            camera_points
        );

        double pose_error =
            cv::norm(
                translation_error_world
            )
            + cv::norm(
                rotation_error_camera
            );

        result.error_history.push_back(
            pose_error
        );

        result.camera_history.push_back(
            t_wc
        );

        result.feature_history.push_back(
            features
        );

        integrate_camera_pose(
            R_wc,
            t_wc,
            v_c,
            dt
        );
    }

    result.R_final =
        R_wc;

    result.t_final =
        t_wc;

    return result;
}

In [ ]:
VSSimulationResult pbvs_result =
    simulate_pbvs(
        points_world,
        R_start,
        t_start,
        R_target,
        t_target,
        0.8,
        0.05,
        300
    );

std::cout
    << "PBVS-Endfehler: "
    << pbvs_result.error_history.back()
    << std::endl;

### IBVS vs. PBVS: Kameratrajektorie

IBVS kontrolliert primär den Bildraum.

PBVS kontrolliert primär den Pose-Raum.

Dadurch unterscheiden sich die resultierenden Trajektorien.

In [ ]:
std::vector<
    std::vector<cv::Vec3d>
> comparison_trajectories = {
    ibvs_result.camera_history,
    pbvs_result.camera_history
};

std::vector<std::string>
    comparison_labels = {
        "IBVS",
        "PBVS"
    };

cv::Mat trajectory_comparison =
    render_camera_trajectories(
        comparison_trajectories,
        comparison_labels,
        t_start,
        t_target,
        "Kameratrajektorie: IBVS vs. PBVS"
    );

show_image(
    trajectory_comparison
)

### PBVS: resultierende Bildtrajektorie

Bei PBVS wird die Pose direkt geregelt.

Die Bewegung der Features im Bildraum ist daher nur ein indirektes Ergebnis der 3D-Bewegung.

In [ ]:
cv::Mat pbvs_feature_plot =
    render_feature_trajectories(
        pbvs_result.feature_history,
        features_target,
        "PBVS: resultierende Feature-Trajektorien"
    );

show_image(
    pbvs_feature_plot
)

## 12. IBVS vs. PBVS

| | IBVS | PBVS |
|---|---|---|
| Regelgröße | Bildmerkmale $\mathbf{s}$ | Pose |
| Fehlerraum | Bildraum | 3D-/Pose-Raum |
| Pose Estimation | nicht zwingend | notwendig |
| Bildtrajektorie | direkt beeinflusst | indirektes Ergebnis |
| 3D-Trajektorie | indirektes Ergebnis | direkt beeinflusst |

Der wesentliche Unterschied ist damit:

> **In welchem Raum wird der Regelfehler definiert und direkt geregelt?**

## 13. Selbstcheck

1. Warum ist eine einmalige Pose-Schätzung noch kein Visual Servoing?
2. Was beschreibt Optical Flow?
3. Was beschreibt die Interaction Matrix?
4. Warum besitzt die Interaction Matrix eines Bildpunktes die Dimension $2\times6$?
5. Welche Anteile hängen von der Tiefe $Z$ ab?
6. Wie werden mehrere Punkt-Jacobians kombiniert?
7. Wie lautet die lokale VS-Strecke?
8. Wie lautet das klassische IBVS-Regelgesetz?
9. Welche Rolle spielt $\lambda$?
10. Warum wird die Pseudoinverse benötigt?
11. Was unterscheidet IBVS und PBVS?

<details>
<summary><b>Kurze Antworten anzeigen</b></summary>

1. Es fehlt die kontinuierliche visuelle Rückführung.
2. Die Bewegung von Bildinformation über die Zeit.
3. Den lokalen Zusammenhang zwischen Kamerageschwindigkeit und Bildbewegung.
4. Ein Punkt liefert zwei Bildgrößen, die Kamera besitzt sechs Bewegungsfreiheitsgrade.
5. Die translatorischen Anteile.
6. Durch vertikales Stapeln.
7. $\mathbf{G}_S(s)=\frac{1}{s}\mathbf{L}_s$.
8. $\mathbf{v}_c=-\lambda\mathbf{L}_s^{+}\mathbf{e}$.
9. Sie bestimmt die Stärke bzw. Geschwindigkeit des Fehlerabbaus.
10. Um aus einem Bildfehler eine geeignete Kamerageschwindigkeit zu bestimmen.
11. IBVS regelt direkt im Bildraum, PBVS direkt im Pose-Raum.

</details>

## 14. Kleine Aufgaben

### Aufgabe A – Kamerabewegung
Verändern Sie `v_test` und testen Sie reine Translationen und Rotationen.

### Aufgabe B – Feature-Geometrie
Verändern Sie die Positionen in `feature_points`.

Was passiert mit dem Rang, wenn die Punkte ungünstig verteilt sind?

### Aufgabe C – Reglerverstärkung
Testen Sie größere Werte von $\lambda$.

Wie verändert sich der Fehlerverlauf?

### Aufgabe D – Tiefenschätzung
Testen Sie beispielsweise `depth_scale = 0.3` und `depth_scale = 3.0`.

Konvergiert der Regler noch?

## 15. Take-away

- Visual Servoing schließt den Regelkreis über die Kamera.
- Optical Flow beschreibt Bewegung im Bild.
- Die Interaction Matrix verbindet Kamerabewegung und Bildbewegung.
- Mehrere Features werden über einen gestapelten Image Jacobian kombiniert.
- Lokal kann die VS-Strecke als $\frac{1}{s}\mathbf{L}_s$ interpretiert werden.
- Das klassische IBVS-Regelgesetz lautet $\mathbf{v}_c=-\lambda\mathbf{L}_s^{+}\mathbf{e}$.
- IBVS regelt direkt im Bildraum.
- PBVS regelt direkt im Pose-Raum.
- Die Wahl des Fehlerraums beeinflusst die resultierende Trajektorie.